# Battery Passport System - Data Exploration

This notebook provides data exploration and analysis for the Battery Passport System project.

## Project Overview

The Battery Passport System monitors electric vehicle (EV) battery health through:
- Real-time sensor measurements (voltage, current)
- Physics-Informed Neural Networks (PINNs)
- Estimation of 8 electrochemical parameters
- State of Charge (SOC) and State of Health (SOH) monitoring

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Import project modules
from src.data_processing.data_loader import BatteryDataProcessor, load_battery_data
from src.battery_model.battery_physics import BatteryModel
from config.config import BATTERY_CONFIG, DATA_CONFIG

print("Libraries imported successfully!")

## 1. Data Loading and Basic Information

In [ ]:
# Load the processed dataset
data_path = '../data/processed/battery_data.csv'
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Display basic statistics
df.describe()

## 2. Data Visualization

In [ ]:
# Plot time series of main variables
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Voltage over time
axes[0, 0].plot(df['Time'], df['Terminal_voltage'], alpha=0.7)
axes[0, 0].set_title('Terminal Voltage vs Time')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Voltage (V)')
axes[0, 0].grid(True, alpha=0.3)

# Current over time
axes[0, 1].plot(df['Time'], df['Current_sense'], alpha=0.7, color='orange')
axes[0, 1].set_title('Current vs Time')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Current (A)')
axes[0, 1].grid(True, alpha=0.3)

# Voltage vs Current
sample_indices = np.random.choice(len(df), size=min(5000, len(df)), replace=False)
axes[1, 0].scatter(df.iloc[sample_indices]['Current_sense'], 
                  df.iloc[sample_indices]['Terminal_voltage'], 
                  alpha=0.5, s=1)
axes[1, 0].set_title('Voltage vs Current')
axes[1, 0].set_xlabel('Current (A)')
axes[1, 0].set_ylabel('Voltage (V)')
axes[1, 0].grid(True, alpha=0.3)

# Voltage derivative
axes[1, 1].plot(df['Time'], df['dV/dt'], alpha=0.7, color='green')
axes[1, 1].set_title('Voltage Derivative vs Time')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('dV/dt (V/s)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Feature Analysis

In [ ]:
# Analyze engineered features
feature_cols = [col for col in df.columns if col not in ['Time', 'Terminal_voltage']]

print(f"Engineered features: {len(feature_cols)}")
print(feature_cols)

In [ ]:
# Plot correlation matrix
correlation_features = ['Terminal_voltage', 'Current_sense', 'Avg_Current', 'Avg_Voltage', 
                       'RMS_Current', 'RMS_Voltage', 'dV/dt', 'dI/dt']

corr_matrix = df[correlation_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key variables
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Voltage distribution
axes[0, 0].hist(df['Terminal_voltage'], bins=50, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Terminal Voltage Distribution')
axes[0, 0].set_xlabel('Voltage (V)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# Current distribution
axes[0, 1].hist(df['Current_sense'], bins=50, alpha=0.7, edgecolor='black', color='orange')
axes[0, 1].set_title('Current Distribution')
axes[0, 1].set_xlabel('Current (A)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# RMS Current distribution
axes[1, 0].hist(df['RMS_Current'], bins=50, alpha=0.7, edgecolor='black', color='green')
axes[1, 0].set_title('RMS Current Distribution')
axes[1, 0].set_xlabel('RMS Current (A)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

# Voltage derivative distribution
axes[1, 1].hist(df['dV/dt'], bins=50, alpha=0.7, edgecolor='black', color='red')
axes[1, 1].set_title('Voltage Derivative Distribution')
axes[1, 1].set_xlabel('dV/dt (V/s)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Battery Model Analysis

In [ ]:
# Analyze the OCV curve
battery_model = BatteryModel()

# Plot OCV curve
soc_range = np.linspace(0, 100, 1000)
ocv_values = [battery_model.calculate_ocv(soc) for soc in soc_range]

plt.figure(figsize=(10, 6))
plt.plot(soc_range, ocv_values, 'b-', linewidth=2)
plt.title('Open Circuit Voltage (OCV) vs State of Charge (SOC)')
plt.xlabel('State of Charge (%)')
plt.ylabel('Open Circuit Voltage (V)')
plt.grid(True, alpha=0.3)

# Mark the data points used for interpolation
plt.scatter(battery_model.ocv_x, battery_model.ocv_y, 
           color='red', s=50, zorder=5, label='Data Points')
plt.legend()
plt.show()

In [ ]:
# Analyze parameter bounds and target values
param_info = BATTERY_CONFIG['parameter_bounds']
target_values = BATTERY_CONFIG['target_values']

param_df = pd.DataFrame({
    'Parameter': list(param_info.keys()),
    'Min_Bound': [bounds[0] for bounds in param_info.values()],
    'Max_Bound': [bounds[1] for bounds in param_info.values()],
    'Target_Value': list(target_values.values())
})

print("Battery Parameter Information:")
print(param_df.to_string(index=False))

In [ ]:
# Visualize parameter ranges
fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(param_df))

# Plot parameter ranges
for i, (_, row) in enumerate(param_df.iterrows()):
    # Use log scale for better visualization
    min_val = np.log10(max(row['Min_Bound'], 1e-6))
    max_val = np.log10(row['Max_Bound'])
    target_val = np.log10(row['Target_Value'])
    
    # Plot range
    ax.plot([min_val, max_val], [i, i], 'b-', linewidth=8, alpha=0.3)
    
    # Plot target value
    ax.plot(target_val, i, 'ro', markersize=8)

ax.set_yticks(y_pos)
ax.set_yticklabels(param_df['Parameter'])
ax.set_xlabel('Log10(Parameter Value)')
ax.set_title('Battery Parameter Bounds and Target Values')
ax.grid(True, alpha=0.3)

# Add legend
ax.plot([], [], 'b-', linewidth=8, alpha=0.3, label='Parameter Range')
ax.plot([], [], 'ro', markersize=8, label='Target Value')
ax.legend()

plt.tight_layout()
plt.show()

## 5. Data Quality Assessment

In [ ]:
# Check for outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Analyze outliers in key variables
key_vars = ['Terminal_voltage', 'Current_sense', 'dV/dt', 'dI/dt']

print("Outlier Analysis:")
print("-" * 50)

for var in key_vars:
    outliers, lower, upper = detect_outliers_iqr(df, var)
    outlier_percentage = (len(outliers) / len(df)) * 100
    
    print(f"{var}:")
    print(f"  Range: [{lower:.6f}, {upper:.6f}]")
    print(f"  Outliers: {len(outliers)} ({outlier_percentage:.2f}%)")
    print()

In [ ]:
# Box plots for outlier visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].boxplot(df['Terminal_voltage'])
axes[0, 0].set_title('Terminal Voltage')
axes[0, 0].set_ylabel('Voltage (V)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].boxplot(df['Current_sense'])
axes[0, 1].set_title('Current Sense')
axes[0, 1].set_ylabel('Current (A)')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].boxplot(df['dV/dt'])
axes[1, 0].set_title('Voltage Derivative')
axes[1, 0].set_ylabel('dV/dt (V/s)')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].boxplot(df['dI/dt'])
axes[1, 1].set_title('Current Derivative')
axes[1, 1].set_ylabel('dI/dt (A/s)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Data Preprocessing for Neural Network

In [ ]:
# Demonstrate data preprocessing
processor = BatteryDataProcessor(window_size=5)

# Get feature information
feature_info = processor.get_feature_info(df)

print("Feature Information:")
print(f"Total features: {len(feature_info['feature_columns'])}")
print(f"Feature columns: {feature_info['feature_columns']}")
print(f"\nDataset shape: {feature_info['shape']}")

In [ ]:
# Prepare data for neural network training
train_data, test_data = processor.prepare_dataset(df, test_size=0.2)

print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

# Show sample of processed data
print("\nSample of training data (first 5 rows, first 10 columns):")
print(train_data[:5, :10])

## 7. Summary and Insights

Based on the data exploration, we can observe:

1. **Data Quality**: The dataset contains comprehensive sensor measurements with engineered features
2. **Feature Engineering**: Multiple derived features (moving averages, RMS, derivatives) provide rich information
3. **Battery Physics**: The OCV curve shows typical lithium-ion battery characteristics
4. **Parameter Ranges**: The target parameters span multiple orders of magnitude, requiring careful normalization
5. **Data Distribution**: Most variables show reasonable distributions suitable for neural network training

This analysis provides the foundation for training physics-informed neural networks to estimate battery parameters accurately.